<a href="https://colab.research.google.com/github/ZoeyZhu2/data_dashboard/blob/main/data_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Goals:
Display buildout progress per bid in RDOF.

Current progress: full map

Desciption:
On a national-scale map (more than 1 state),
All census block groups depicted as hexagons.
Display baseline data (June 2022 from NBM) of census block groups served colored dark.
Display planned census block groups served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid.
Display newly served (cumulative) census block groups up to a particular time (July 2022 to December 2025) colored in a dark color.
Color each census block group with saturation corresponding to percentage served.
Every bid should get the same color.


State-scale map:
Hexagons represent fewer and fewer RDOF locations as you zoom in.
Display baseline data (June 2022 from NBM) of RDOF locations served colored dark.
Display planned RDOF locations served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid (each bid gets its own color)
Display newly served (cumulative) RDOF locations up to a particular time (July 2022 to December 2025) colored in a dark color that corresponds to that particular RDOF bid.


In 2 table below the map,
By CBG or RDOF location:
Show the bid awardee (company),
number of planned CBGs/location,
number of currently built out CBGs/locations (cumulative) that corresponds to the time chosen
number of baseline CBGs/locations, and total number of CBGs/locations.


Above the map, toggle options for which state to show data (select 1-50), toggle options for which companies/bids to show data (1 to all), and a slider from June 2022 to December 2025 in 1 month increments to select the "current time" to show cumulative progress.

In [1]:
# REQUIRED: authenticate with GCP project
GCP_PROJECT_ID = "broadband-data"  #@param {type:"string"}

from google.colab import auth
from google.colab import userdata

auth.authenticate_user(project_id=GCP_PROJECT_ID)

In [2]:
# Authenticate GCP Client
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT_ID)
print("Authenticated & Output Directory Initialized!")

Authenticated & Output Directory Initialized!


In [6]:
# Cell 1: querying for one row per RDOF location, ALL states
import os, pandas as pd
from google.cloud import bigquery

BASELINE = "2022_06"       # baseline time
SERVED_THRESH = 0.80       # share of locations for a hexagon / block group to count as "served"
STATES = None
MAX_GB = 800               # safety cap for BigQuery
CACHE = ("rdof_locations_all.parquet" if STATES is None
         else "rdof_locations_" + "_".join(STATES) + ".parquet")   # delete the file to re-query


# not including defaulted census blocks
# div block_geoid by 1000 to get the census block group number
SQL = """
WITH rdof_blocks AS (
  SELECT DISTINCT census_block, applicant, LTRIM(frn, '0') AS rdof_frn, sac
  FROM `broadband-data.rdof.applicants_hist_2023_cb`
  WHERE state_abbr IN UNNEST(@states)
    AND census_block NOT IN (SELECT census_block FROM `broadband-data.rdof.defaults_post_auth_cb`)
),
rdof_locs AS (
  SELECT f.location_id, f.state, DIV(f.block_geoid, 1000) AS cbg, f.latitude, f.longitude, f.h3_9,
         b.applicant AS awardee, b.rdof_frn, b.sac
  FROM `broadband-data.bsl_fabric.v7` f
  JOIN rdof_blocks b ON f.block_geoid = b.census_block
  WHERE f.bsl_flag AND f.state IN UNNEST(@states)
),
first_served AS (
  SELECT l.location_id, MIN(a.period) AS first_served_period
  FROM rdof_locs l
  JOIN `broadband-data.fcc_bdc.nbm_hive` a
    ON a.location_id = l.location_id
   AND a.state_usps = l.state
   AND LTRIM(a.frn, '0') = l.rdof_frn
  WHERE a.state_usps IN UNNEST(@states)
   -- optional speed filter: AND a.max_advertised_download_speed >= 100
   --                        AND a.max_advertised_upload_speed >= 20
  GROUP BY l.location_id
)
SELECT l.*, fs.first_served_period
FROM rdof_locs l
LEFT JOIN first_served fs USING (location_id)
"""

# pick states to use
all_states = client.query("""SELECT DISTINCT state_abbr
    FROM `broadband-data.rdof.applicants_hist_2023_cb` ORDER BY 1""").to_dataframe().state_abbr.tolist()
states_to_load = STATES or all_states
params = [bigquery.ArrayQueryParameter("states", "STRING", states_to_load)]
print(len(states_to_load), "states:", ", ".join(states_to_load))

if os.path.exists(CACHE):
    loc = pd.read_parquet(CACHE)
    print("Loaded from cache:", CACHE)
else:
    dry = client.query(SQL, job_config=bigquery.QueryJobConfig(
        query_parameters=params, dry_run=True, use_query_cache=False))
    print(f"Estimated scan: {dry.total_bytes_processed / 1e9:,.0f} GB "
          "(may show 0 for external tables, so the cap below is the real safety net)")
    loc = client.query(SQL, job_config=bigquery.QueryJobConfig(
        query_parameters=params, maximum_bytes_billed=int(MAX_GB * 1e9))).to_dataframe()
    loc["latitude"] = loc.latitude.astype("float32")
    loc["longitude"] = loc.longitude.astype("float32")
    loc.to_parquet(CACHE)
    print("Saved cache:", CACHE)

periods = sorted(client.query(f"""
    SELECT DISTINCT period FROM `broadband-data.fcc_bdc.nbm_hive`
    WHERE state_usps = '{states_to_load[0]}'""").to_dataframe().period)

summary = loc.groupby("state").agg(
    locations=("location_id", "size"), block_groups=("cbg", "nunique"),
    awardees=("awardee", "nunique"), ever_served=("first_served_period", "count"))
print(f"\n{len(loc):,} locations | {loc.awardee.nunique()} awardees | periods {periods[0]} to {periods[-1]}")
summary

49 states: AL, AR, AZ, CA, CO, CT, DE, FL, GA, HI, IA, ID, IL, IN, KS, KY, LA, MA, MD, ME, MI, MN, MO, MP, MS, MT, NC, ND, NE, NH, NM, NV, NY, OH, OK, OR, PA, RI, SC, SD, TN, TX, UT, VA, VT, WA, WI, WV, WY
Estimated scan: 6 GB (may show 0 for external tables, so the cap below is the real safety net)
Saved cache: rdof_locations_all.parquet

1,814,347 locations | 345 awardees | periods 2022_06 to 2025_12


,locations,block_groups,awardees,ever_served
state,,,,
AL,82226,516,17,13204
AR,82713,570,20,48098
AZ,41447,448,4,5286
CA,10426,118,5,1744
CO,16838,174,5,3603
CT,110,3,1,0
DE,6813,136,1,0
FL,41799,289,8,12510
GA,81125,642,13,36089


In [ ]:
# Cell 2: block group table, hexagons, status rules, colors
!pip -q install "h3>=4"
import h3, colorsys, numpy as np, pandas as pd

CBG_RES = 6            # size of block group hexagons: 5 = big, 6 = medium, 7 = small

PIDX = {p: i for i, p in enumerate(periods)}      # period -> 0, 1, 2 ...
BASE_I = PIDX[BASELINE]

if "first_served_period" in loc.columns:          # compact storage; safe to re-run
    loc["fs_i"] = loc.first_served_period.map(PIDX).fillna(99).astype("int8")   # 99 = never served
    loc = loc.drop(columns=["first_served_period"])
for c in ("state", "awardee"):
    loc[c] = loc[c].astype("category")

# --- block groups: location counts, cumulative served per period (computed once) ---
cnt = (loc.groupby(["state", "awardee", "sac", "cbg", "fs_i"], observed=True)
          .size().unstack("fs_i", fill_value=0))
cbg_tab = (loc.groupby(["state", "awardee", "sac", "cbg"], observed=True)
              .agg(n_total=("location_id", "size"), lat=("latitude", "mean"), lon=("longitude", "mean")))
cum_np = (cnt.reindex(columns=range(len(periods)), fill_value=0).cumsum(axis=1)
             .reindex(cbg_tab.index).to_numpy())          # served by period p = first served <= p
cbg_tab["n_base"] = cum_np[:, BASE_I]
bid_states = cbg_tab.reset_index()[["state", "awardee"]].drop_duplicates()   # which bids are in which state

# --- one distinct hexagon per block group, placed once ---
def place_hexes(info):
    occupied, cells = set(), {}
    order = info.sort_values("n_total", ascending=False)
    # zip over plain arrays (iterrows builds a Series per row and is very slow)
    for key, la, lo in zip(order.index.tolist(), order.lat.to_numpy(), order.lon.to_numpy()):
        home = h3.latlng_to_cell(la, lo, CBG_RES)
        cell, k = home, 0
        while cell in occupied:
            k += 1
            free = [c for c in h3.grid_ring(home, k) if c not in occupied]
            if free:
                cell = min(free, key=lambda c, la=la, lo=lo: (h3.cell_to_latlng(c)[0] - la) ** 2
                                                           + (h3.cell_to_latlng(c)[1] - lo) ** 2)
        occupied.add(cell)
        cells[key] = cell
    return cells

CBG_HEX = place_hexes(cbg_tab)             # key: (state, awardee, sac, cbg) -> hexagon id
cbg_tab["hex"] = [CBG_HEX[k] for k in cbg_tab.index]     # attach once, so the slider never loops over it

_geom = {}
def geom_of(h):
    if h not in _geom:
        ring = [[round(lng, 5), round(lat, 5)] for lat, lng in h3.cell_to_boundary(h)]  # GeoJSON: [lng, lat]
        ring.append(ring[0])
        _geom[h] = {"type": "Polygon", "coordinates": [ring]}
    return _geom[h]

# --- colors ---
# One hue per bid, in a fixed order, reassigned per map from the bids on screen.
# TODO: reads well for ~8 bids on screen at once; hues repeat beyond that.
BASE_HEX = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
def hue_map(bid_list):
    return {b: colorsys.rgb_to_hls(*[int(BASE_HEX[i % 8][j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]
            for i, b in enumerate(bid_list)}
HUE = {}
LIGHTNESS = {"planned": 0.82, "new": 0.58, "baseline": 0.34}   # light / medium / darkest
LABEL = {"planned": "Planned", "new": "Newly served (cumulative)",
         "baseline": f"Baseline (served by {BASELINE})"}
NATIONAL_HUE = colorsys.rgb_to_hls(*[int("#2a78d6"[j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]

def shade(awardee, cls, pct, single_color=False):
    """One color. Used for the legend swatches; the map uses shade_col below (same result, vectorized).
    Hue = bid (or one shared blue if single_color), lightness = status,
    saturation = share of locations served (20%-100%)."""
    hue = NATIONAL_HUE if single_color else HUE[awardee]
    r, g, b = colorsys.hls_to_rgb(hue, LIGHTNESS[cls], 0.20 + 0.80 * float(pct))
    return "#%02x%02x%02x" % (int(r * 255), int(g * 255), int(b * 255))

def hls_to_hex(h, l, s):
    """colorsys.hls_to_rgb for whole arrays at once."""
    h, l, s = (np.asarray(x, float) for x in (h, l, s))
    m2 = np.where(l <= 0.5, l * (1 + s), l + s - l * s)
    m1 = 2 * l - m2
    def ch(hh):
        hh = hh % 1.0
        return np.where(hh < 1 / 6, m1 + (m2 - m1) * hh * 6,
               np.where(hh < 0.5, m2,
               np.where(hh < 2 / 3, m1 + (m2 - m1) * (2 / 3 - hh) * 6, m1)))
    rgb = (np.stack([ch(h + 1 / 3), ch(h), ch(h - 1 / 3)], axis=1) * 255).astype(int)
    return ["#%02x%02x%02x" % tuple(r) for r in rgb.tolist()]

def shade_col(t, single_color=False):
    """Colors for every row of a hexagon table `t` in one go."""
    hue = (np.full(len(t), NATIONAL_HUE) if single_color
           else t.awardee.astype(str).map(HUE).to_numpy(float))
    light = t.cls.map(LIGHTNESS).to_numpy(float)
    return hls_to_hex(hue, light, 0.20 + 0.80 * t.pct.to_numpy(float))

# --- status rules ---
def classify(t):
    t["pct"] = t.n_served / t.n_total
    t["pct_base"] = t.n_base / t.n_total
    t["cls"] = np.where(t.pct_base >= SERVED_THRESH, "baseline",
                np.where(t.pct >= SERVED_THRESH, "new", "planned"))
    return t

def block_group_table(states, companies, p):
    pi = PIDX[p]
    idx = cbg_tab.index
    m = idx.get_level_values("state").isin(states) & idx.get_level_values("awardee").isin(companies)
    t = cbg_tab[m].reset_index()                      # already carries the "hex" column
    t["n_served"] = cum_np[m, pi]
    t["area"] = t.cbg.astype(str).str.zfill(12)
    return classify(t)

# --- location hexagons at any size, computed fast from integer H3 ids ---
H3_INT = np.fromiter((int(c, 16) for c in loc.h3_9), dtype=np.uint64, count=len(loc))

def parent_int(cells, res):
    """H3 parent at `res` for an array of resolution-9 cell ids (pure bit arithmetic)."""
    keep = ~np.uint64(0xF << 52)
    return (cells & keep) | np.uint64(res << 52) | np.uint64((1 << (3 * (15 - res))) - 1)

# computed once, so panning/zooming only has to index into plain integer arrays
PARENT = {r: parent_int(H3_INT, r) for r in (5, 6, 7, 8, 9)}   # uses ~40 MB per million locations
AW_CODE = loc.awardee.cat.codes.to_numpy()
SAC, FS = loc.sac.to_numpy(), loc.fs_i.to_numpy()
LAT, LON = loc.latitude.to_numpy(), loc.longitude.to_numpy()
AW_NAMES = loc.awardee.cat.categories

def location_table(idx, p, res):
    """Hexagons of size `res` for the locations at row positions `idx` (the map viewport)."""
    pi = PIDX[p]
    d = pd.DataFrame({"aw": AW_CODE[idx], "sac": SAC[idx], "cell": PARENT[res][idx],
                      "sp": FS[idx] <= pi, "sb": FS[idx] <= BASE_I,
                      "lat": LAT[idx], "lon": LON[idx]})
    t = (d.groupby(["aw", "sac", "cell"], sort=False)
          .agg(n_total=("sp", "size"), n_served=("sp", "sum"), n_base=("sb", "sum"),
               lat=("lat", "mean"), lon=("lon", "mean")).reset_index())
    t["awardee"] = AW_NAMES[t.aw.to_numpy()]
    t["hex"] = [format(c, "x") for c in t.cell.tolist()]
    t["area"] = t.hex
    return classify(t)

def filing_for(month):
    """Slider month 'YYYY-MM' -> latest NBM filing at or before it ('YYYY_MM')."""
    return [p for p in periods if p.replace("_", "-") <= month][-1]

In [ ]:
# ---- Cell 3: interactive map (zoom in for locations) + tables + toggles ----
!pip -q install ipyleaflet
import asyncio, threading
import ipywidgets as widgets, numpy as np, pandas as pd
from IPython.display import display, clear_output
from ipyleaflet import Map, GeoJSON, TileLayer, WidgetControl
try:
    from google.colab import output
    output.enable_custom_widget_manager()      # lets Colab show ipyleaflet
except Exception:
    pass

# ---- settings you can tune ----
MAX_VIEW_HEXES = 15000      # more hexagons than this in view -> keep block groups
MAX_VIEW_LOCS = 2_000_000   # more locations than this in view -> keep block groups
MAX_BID_COLORS = 8          # per-bid colors only when the view has this many bids or fewer
MARGIN = 0.15               # draw this much extra on each side of the screen (0 = none); bigger = fewer redraws
REDRAW_DELAY = 0            # seconds the map must stop moving before hexagons redraw (0 = redraw immediately)
HOVER_DELAY = 0.01          # seconds to rest on a hexagon before the info box updates (0 = immediately)

log_out = widgets.Output()      # any error inside a map/hover callback shows up here, under the map

def _logged(fn):
    def wrapper(*a, **k):
        try:
            return fn(*a, **k)
        except Exception:
            import traceback
            with log_out:
                clear_output(wait=True)
                print(traceback.format_exc())
    return wrapper

def debounced(fn, delay):
    """Wrap fn so it only runs after `delay` seconds with no further calls (delay 0 = run right away)."""
    fn = _logged(fn)
    if not delay:
        return fn
    h = {"t": None}
    def run(a, k):
        h["t"] = None
        fn(*a, **k)
    def wrapper(*a, **k):
        if h["t"] is not None:
            h["t"].cancel()
        try:
            h["t"] = asyncio.get_running_loop().call_later(delay, run, a, k)
        except RuntimeError:                       # no running loop: fall back to a thread timer
            h["t"] = threading.Timer(delay, run, args=(a, k)); h["t"].start()
    return wrapper

def loc_res_for(z):
    """Hexagon size by zoom: coarse when zoomed out, address-sized when zoomed in."""
    z = int(z)
    if z >= 11:
        return 9
    return {10: 8, 9: 7, 8: 6, 7: 5}.get(z)      # None below zoom 7 -> block groups

# If the background tiles are blocked in your notebook, remove the TileLayer below.
TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATTR = "Tiles © Esri, HERE, Garmin, © OpenStreetMap contributors"
COLS = ["planned", "new", "baseline"]
LAT, LON = loc.latitude.to_numpy(), loc.longitude.to_numpy()

months = [str(x) for x in pd.period_range(BASELINE.replace("_", "-"),
                                          periods[-1].replace("_", "-"), freq="M")]
states = sorted(bid_states.state.astype(str).unique())
companies = sorted(bid_states.awardee.astype(str).unique())

def _bids_in(sel):
    return sorted(bid_states[bid_states.state.astype(str).isin(sel)].awardee.astype(str).unique())

# ---------------- the map ----------------
m = Map(center=(39.0, -90.0), zoom=4, scroll_wheel_zoom=True,
        layout=widgets.Layout(height="650px"))
m.layers = (TileLayer(url=TILES, attribution=ATTR),)

info = widgets.HTML("<div style='font:12px sans-serif;background:#fff;padding:6px 8px;"
                    "border:1px solid #ccc;border-radius:4px'>Hover over a hexagon</div>")
legend = widgets.HTML("")
legend.layout.display = "none"
m.add_control(WidgetControl(widget=info, position="topright"))
m.add_control(WidgetControl(widget=legend, position="bottomleft"))

def _show_info(p):
    info.value = ("<div style='font:12px sans-serif;background:#fff;padding:6px 8px;"
                  "border:1px solid #ccc;border-radius:4px'>"
                  f"<b>{p['bid']}</b><br>{p['status']}<br>{p['pct_served']}% served "
                  f"({p['n_served']:,} of {p['n_total']:,} locations)<br>"
                  f"<span style='color:#777'>{p['area']}</span></div>")

_show_info_later = debounced(_show_info, HOVER_DELAY)

def on_hover(feature=None, **kwargs):
    if feature:
        _show_info(feature["properties"])

def make_layer(t):
    feats = [{"type": "Feature", "geometry": geom_of(h),
              "properties": {"bid": a, "status": LABEL[c], "pct_served": pc,
                             "n_served": ns, "n_total": nt, "area": ar, "color": co}}
             for h, a, c, pc, ns, nt, ar, co in zip(
                 t.hex.tolist(), t.awardee.astype(str).tolist(), t.cls.tolist(),
                 (100 * t.pct).round(1).tolist(), t.n_served.astype(int).tolist(),
                 t.n_total.astype(int).tolist(), t.area.tolist(), t.color.tolist())]
    layer = GeoJSON(
        data={"type": "FeatureCollection", "features": feats},
        hover_style={"weight": 2, "color": "#222222"},
        style_callback=lambda f: {"fillColor": f["properties"]["color"], "fillOpacity": 0.9,
                                  "color": "#ffffff", "weight": 0.4})
    layer.on_hover(on_hover)
    return layer

_current = {"layer": None}
def show_layer(layer):
    if _current["layer"] is layer:
        return
    if _current["layer"] is not None:
        m.remove_layer(_current["layer"])
    m.add_layer(layer)
    _current["layer"] = layer

S = {"p": None, "cb": None, "bg_layer": None, "sel_mask": None, "rendered": None}
note = widgets.HTML("")
banner = widgets.HTML("")
table_out = widgets.Output()

def swatches(awardee):
    return " ".join(
        f'<span style="color:{shade(awardee, c, 1.0, False)}">&#9632;</span>{LABEL[c].split(" (")[0]}'
        for c in ("baseline", "new", "planned"))

STABLE_IDX = {}                              # company -> color slot, kept for the whole session
HUE_OF = hue_map(list(range(8)))             # slot -> hue

def assign_hues(bids):
    """A company keeps its color; a new company takes a slot not used by the others in view."""
    for b in bids:
        if b not in STABLE_IDX:
            taken = {STABLE_IDX[x] for x in bids if x in STABLE_IDX}
            STABLE_IDX[b] = next((i for i in range(8) if i not in taken), len(STABLE_IDX) % 8)
    HUE.clear()
    HUE.update({b: HUE_OF[STABLE_IDX[b]] for b in bids})

def update_detail(change=None):
    """Block groups when zoomed far out; otherwise location hexagons sized to the zoom level.
    Draws a margin around the screen and skips redrawing while the view stays inside it."""
    if S["bg_layer"] is None:
        return
    legend.layout.display = "none"
    res = (loc_res_for(m.zoom) if view_w.value == "Auto"
           and m.bounds is not None and len(m.bounds) == 2 else None)
    if res is None:
        S["rendered"] = None
        show_layer(S["bg_layer"])
        note.value = (f"<i>Showing block groups. Zoom in to level 7 or closer (now {int(m.zoom)}) "
                      "to see locations.</i>" if view_w.value == "Auto"
                      else "<i>Showing block groups.</i>")
        return
    (s, w), (n, e) = m.bounds
    R = S.get("rendered")
    if R and R["res"] == res and R["s"] <= s and n <= R["n"] and R["w"] <= w and e <= R["e"]:
        return                                    # view still inside what is already drawn
    dlat, dlon = MARGIN * (n - s), MARGIN * (e - w)
    s, n, w, e = s - dlat, n + dlat, w - dlon, e + dlon
    idx = np.flatnonzero(S["sel_mask"] & (LAT >= s) & (LAT <= n) & (LON >= w) & (LON <= e))
    if len(idx) == 0:
        S["rendered"] = None
        show_layer(S["bg_layer"]); note.value = "<i>No RDOF locations in view.</i>"; return
    if len(idx) > MAX_VIEW_LOCS:
        S["rendered"] = None
        show_layer(S["bg_layer"])
        note.value = f"<i>{len(idx):,} locations in view. Zoom in to see them; showing block groups.</i>"
        return
    t = location_table(idx, S["p"], res)
    if len(t) > MAX_VIEW_HEXES:
        S["rendered"] = None
        show_layer(S["bg_layer"])
        note.value = (f"<i>{len(t):,} hexagons in view. Zoom in further to see locations; "
                      "showing block groups.</i>")
        return
    bids = sorted(t.awardee.astype(str).unique())
    per_bid = len(bids) <= MAX_BID_COLORS
    if per_bid:
        assign_hues(bids)
    t["color"] = shade_col(t, not per_bid)
    show_layer(make_layer(t))
    S["rendered"] = dict(res=res, s=s, n=n, w=w, e=e)
    what = "individual locations" if res == 9 else "groups of locations"
    note.value = (f"<i>Showing {len(t):,} hexagons of {what} in view (zoom {int(m.zoom)}). "
                  "Zoom in further for single addresses.</i>" if res < 9
                  else f"<i>Showing {len(t):,} location hexagons in view (zoom {int(m.zoom)}).</i>")
    if per_bid:
        legend.value = ("<div style='font:12px sans-serif;background:#fff;padding:8px 10px;"
                        "border:1px solid #ccc;border-radius:6px;max-height:40vh;overflow-y:auto'>"
                        + "".join(f"<div><b>{b}</b> {swatches(b)}</div>" for b in bids)
                        + "<div><i>Saturation = % served</i></div></div>")
        legend.layout.display = "block"

m.observe(debounced(update_detail, REDRAW_DELAY), names=["bounds"])

# ---------------- tables and banner ----------------
def banner_html(sel, n_shown, n_avail):
    chip = ('<span style="display:inline-block;margin:0 4px 4px 0;padding:2px 10px;border-radius:12px;'
            'font:600 13px sans-serif;{}">{}</span>')
    on  = "".join(chip.format("background:#1f7a3a;color:#fff", s) for s in sel) or "<i>none</i>"
    off = "".join(chip.format("background:#e6e6e6;color:#777;text-decoration:line-through", s)
                  for s in states if s not in sel) or "<i>none</i>"
    return (f'<div style="font:13px sans-serif;margin:6px 0"><b>Showing:</b> {on} '
            f'&nbsp;&nbsp; <b>Not showing:</b> {off} &nbsp;&nbsp; '
            f'<b>Companies:</b> {n_shown} of {n_avail}</div>')

def show_tables(cb, group):
    cb = cb.copy()
    cb["planned_n"] = cb.n_total - cb.n_served
    cb["new_n"] = cb.n_served - cb.n_base
    bg = (cb.pivot_table(index=group, columns="cls", values="cbg", aggfunc="count",
                         fill_value=0, observed=True).reindex(columns=COLS, fill_value=0))
    lc = cb.groupby(group, observed=True)[["planned_n", "new_n", "n_base"]].sum()
    for tab in (bg, lc):
        tab.columns = ["Planned", "Built since baseline (cumulative)", "Baseline"]
        tab["Total"] = tab.sum(axis=1)
        if group == "state":
            tab.loc["All states"] = tab.sum()
    with table_out:
        clear_output(wait=True)
        print("Block groups"); display(bg)
        print("Locations");    display(lc)

# ---------------- refresh when a control changes ----------------
def refresh(fit=False):
    sel = [s for s in states_text.value.split(",") if s]
    comp = list(company_w.value)
    avail = _bids_in(sel)
    banner.value = banner_html(sel, len([c for c in comp if c in set(avail)]), len(avail))
    S["bg_layer"] = None
    S["rendered"] = None
    if not sel or not comp:
        if _current["layer"] is not None:
            m.remove_layer(_current["layer"]); _current["layer"] = None
        note.value = "<i>Select at least one state and one company.</i>"
        with table_out:
            clear_output()
        return
    p = filing_for(month_w.value)
    cb = block_group_table(sel, comp, p)
    if cb.empty:
        note.value = "<i>No RDOF block groups for this selection.</i>"; return
    cb["color"] = shade_col(cb, True)                      # one shared color
    S.update(p=p, cb=cb, bg_layer=make_layer(cb), rendered=None,
             sel_mask=(loc.state.isin(sel) & loc.awardee.isin(comp)).to_numpy())
    show_tables(cb, "state" if len(sel) > 1 else "awardee")
    if fit:
        try:
            m.fit_bounds([[float(cb.lat.min()), float(cb.lon.min())],
                          [float(cb.lat.max()), float(cb.lon.max())]])
        except Exception:
            m.center = (float(cb.lat.mean()), float(cb.lon.mean())); m.zoom = 6
    update_detail()

# ---------------- controls ----------------
state_boxes = {s: widgets.ToggleButton(value=True, description=s, icon="check",
                                       button_style="success", tooltip=f"Show/hide {s}",
                                       layout=widgets.Layout(width="70px"))
               for s in states}
all_btn  = widgets.Button(description="All",  layout=widgets.Layout(width="70px"))
none_btn = widgets.Button(description="None", layout=widgets.Layout(width="70px"))
states_text = widgets.Text(value=",".join(states), layout=widgets.Layout(display="none"))  # hidden carrier
company_w = widgets.SelectMultiple(options=companies, value=tuple(companies),
                                   description="Companies", rows=min(max(len(companies), 2), 8),
                                   layout=widgets.Layout(width="420px"))
view_w = widgets.Dropdown(options=["Auto", "Block groups only"], value="Auto",
                          description="View", layout=widgets.Layout(width="320px"))
month_w = widgets.SelectionSlider(options=months, value=months[-1], description="As of",
                                  continuous_update=False,       # redraw on release, not on every step
                                  layout=widgets.Layout(width="520px"))

_bulk = False
_syncing = False
def _current_states():
    return ",".join(s for s, b in state_boxes.items() if b.value)

def on_toggle(change):
    b = change["owner"]
    b.button_style = "success" if b.value else ""
    b.icon = "check" if b.value else ""
    if not _bulk:
        states_text.value = _current_states()

def set_all(value):
    global _bulk
    _bulk = True
    for b in state_boxes.values():
        b.value = value
    _bulk = False
    states_text.value = _current_states()

def sync_companies(change=None):
    """Companies list = bids in the selected state(s); select all of them, then redraw."""
    global _syncing
    sel = [s for s in states_text.value.split(",") if s]
    opts = _bids_in(sel)
    _syncing = True
    company_w.value = ()
    company_w.options = opts
    company_w.rows = min(max(len(opts), 2), 8)
    company_w.value = tuple(opts)
    _syncing = False
    refresh(fit=True)

for b in state_boxes.values():
    b.observe(on_toggle, "value")
all_btn.on_click(lambda _: set_all(True))
none_btn.on_click(lambda _: set_all(False))
states_text.observe(sync_companies, "value")
company_w.observe(lambda c: None if _syncing else refresh(fit=True), "value")
month_w.observe(lambda c: refresh(fit=False), "value")
view_w.observe(update_detail, "value")

display(widgets.VBox([
    widgets.HBox([widgets.Label("States:"), all_btn, none_btn]),
    widgets.HBox(list(state_boxes.values()), layout=widgets.Layout(flex_flow="row wrap")),
    company_w, view_w, month_w, banner, m, note, log_out, table_out, states_text]))
sync_companies()